In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import time
import numpy as np
import scipy

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams["figure.figsize"] = [12,8]
from matplotlib.patches import Arc, FancyArrowPatch

from scipy.special import logsumexp

# Lighthouse

This problem is taken from "Data Analysis, a Bayesian Tutorial" by D.S. Silva with J. Skiling. A lighthouse distance $h=1$ from the shore is rotating with constant angular frequency and emitting thin beams of light at random. The probability of emission is uniform in time. The signals are picked up on the shore by an array of detectors and their location is saved in the file `lighthouse.txt`.  Both the  horizontal location of the lighthouse $x_{lh}$ and its distance from the shore $h$ are unknown. The task is to estimate those positions.

In [ ]:
flash_x = np.loadtxt('lighthouse_2d.txt')

## Sampling distribution

As we have already established the sampling distribution $P(x|x_{lh},h)$ is given by the Cauchy distribution

$$P(x|x_{lh},h)=\frac{h}{\pi}\frac{1}{\left(x-x_{lh}\right)^2+h^2}$$

## Problem 1

Write down function that implements this probability distribution function (pdf)

In [ ]:
def p(x_lh, h, x):
    return (h / np.pi) * (1 / ((x - x_lh)**2 + h**2))

and the logarithm of the pdf

In [ ]:
def log_p(x_lh, h, x):
    return np.log(h) - np.log(np.pi) - np.log((x - x_lh)**2 + h**2)

## Problem 2

Calculate the posterior distribution on $x_{lh}$ and $h$ after detection of one flash.
Assume a uniform prior on some interval for both $x_{lh}$ and $h$. Choose the intervals yourself.

Calculate the MAP estimate for $x_{lh}$ and $h$. Then calculate the marginal distributions for $x_{lh}$ and $h$ and plot them. Calculate the MAP estimates for each distributions separately.

In [ ]:
x_min = -20; x_max=20;
h_min = 1.0; h_max= 10;

The distribution will be represented as an array with columns corresponding to the values of $x_{lh}$ and rows to values of $h$. Start by discretizing the intervals you have chosen for $x_{lh}$ and $h$. Use the `np.linspace` function for that

In [ ]:
nx = 4000
ny = 3000
x_lhs = np.linspace(x_min, x_max, nx)
hs = np.linspace(h_min, h_max, ny)

I suggest to use different values for each dimensions. In that way if you by mistake exchange row and columns you will get an error. The prior will be a constant and we do not have to worry about the normalization (we will normalize the posterior) so we will set this value to one

In [ ]:
prior_array = np.ones((ny,nx))

The next step will be to calculate the value of $p(x_0|x_{lh},h)$ for each value $x_{ls}$ and $h$ in arrays `x_lhs` and `hs`, where $x_0$ is the the position of the first flash `flash_x[0]`. This could be done by a simple double loop

In [ ]:
p_array = np.empty((ny,nx))
p_array.shape

In [ ]:
start_time = time.time()
for i, x_lh in enumerate(x_lhs):
  for j, h in enumerate(hs):
    p_array[j,i]=p(x_lh,h,flash_x[0])
end_time = time.time()
elapsed_time_loop = end_time-start_time
print(f"Elapsed time = {elapsed_time_loop:.1f}s")

However this is not recommended, as the explicit loops in Python tend to be slow. The proper way is to construct a grid of $x_{lh}$ and $h$ values using function [`numpy.meshgrid`](https://numpy.org/doc/stable/reference/generated/numpy.meshgrid.html).

In [ ]:
start_time=time.time()
x_lh_grid, h_grid = np.meshgrid(x_lhs,hs)
end_time = time.time()
elapsed_time_meshgrid = end_time-start_time
print(f"Elapsed time = {elapsed_time_meshgrid:.3f}s")

Array `x_lh_grid` contains the values of $x_{lh}$ for every cell of array `p_array`. It consists of identical rows

In [ ]:
x_lh_grid[0]

In [ ]:
x_lh_grid[1]

Similarly the array `h_grid` contains the values of $h$ for every cell in array `p_array` and consists of identical columns

In [ ]:
h_grid[:,0]

In [ ]:
h_grid[:,1]

Now we can compute the `p_array` with a single call to `p`

In [ ]:
start_time=time.time()
p_array = p(x_lh_grid, h_grid, flash_x[0])
end_time = time.time()
elapsed_time_meshgrid_p = end_time-start_time
print(f"Elapsed time = {elapsed_time_meshgrid_p:.3f}s")

The whole computation took

In [ ]:
print(f"{(elapsed_time_meshgrid+elapsed_time_meshgrid_p)*1000:.0f}ms")

compared to

In [ ]:
print(f"{(elapsed_time_loop):.1f}s")

for the explicit loop, so the speedup is

In [ ]:
print(f"{elapsed_time_loop/(elapsed_time_meshgrid+elapsed_time_meshgrid_p):.1f}")

times. This is not very important here, as four seconds is still a very short time, but this may quickly become an issue when we add more data.

The array can be visualized using the `imshow` function, but first we will normalize  the posterior so the sum (not integral) is equal to one.

In [ ]:
posterior1 = p_array*prior_array
Z = posterior1.sum()
posterior1/=Z

In [ ]:
plt.imshow(posterior1, origin='lower', extent=(x_min, x_max, h_min, h_max), aspect='auto');
plt.colorbar();

but we can get a clearer picture using the `contour` and `contourf` functions

In [ ]:
levels=20
cs = plt.contourf(x_lhs,hs, posterior1, levels=levels);
plt.contour(x_lhs,hs, posterior1, colors='black', linewidths=0.5, levels=levels);
plt.colorbar(cs);

Depending on the interval that you have chosen for the prior, you will probably not see  anything on the plots. This is OK as for one flash the distribution is strongly peaked around $(x_0,0)$. This will get better when we obtain more data.

#### MAP estimate

__Hint:__ Use the `numpy.argmax` and `numpy.unravel_index` functions.

In [ ]:
map_idx = np.unravel_index(np.argmax(posterior1), posterior1.shape)
map_h = hs[map_idx[0]]
map_x = x_lhs[map_idx[1]]
print(f"Joint MAP: x_lh = {map_x}, h = {map_h}")

### Marginal distributions

You can obtain the marginal distribution $p(x_{lh}|x_0)$ by summing the array along the axis zero.

Similarly the distribution $p(h|x_0)$ will be obtained by summing  over the axis one.

In [ ]:
marg_x = posterior1.sum(axis=0)
marg_h = posterior1.sum(axis=1)

map_marg_x = x_lhs[np.argmax(marg_x)]
map_marg_h = hs[np.argmax(marg_h)]
print(f"Marginal MAP: x_lh = {map_marg_x}, h = {map_marg_h}")

plt.plot(x_lhs, marg_x)
plt.xlabel('x_lh')
plt.ylabel('Marginal Probability')
plt.show()

plt.plot(hs, marg_h)
plt.xlabel('h')
plt.ylabel('Marginal Probability')
plt.show()

### Logarithms

More often than not we will be using logarithm of probabilities. Perform same calculations as previously, using the log of the probability.

In [ ]:
log_prior_array = np.zeros_like(prior_array)

In [ ]:
log_p_array = log_p(x_lh_grid, h_grid, flash_x[0])
log_posterior1= log_p_array+log_prior_array
Z= logsumexp(log_posterior1)
log_posterior1 -= Z

In [ ]:
max_log_posterior = log_posterior1.max()
cs = plt.contourf(x_lhs,hs, log_posterior1-max_log_posterior);
plt.contour(x_lhs,hs, log_posterior1-max_log_posterior,colors = 'black', linestyles='-');
plt.colorbar(cs);

Because logarithm is a monotonically increasing function we can calculate the MAP estimate in the same way as for the  probabilities.

#### Marginal distributions

Again we have to partially sum the `log_posterior1` array. But before doing that we have to exponentiate it. To avoid numerical problems it is best to use the `scipy.special.logsumexp` function.

In [ ]:
map_log_idx = np.unravel_index(np.argmax(log_posterior1), log_posterior1.shape)
print(f"Log Joint MAP: x_lh = {x_lhs[map_log_idx[1]]}, h = {hs[map_log_idx[0]]}")

log_marg_x = logsumexp(log_posterior1, axis=0)
log_marg_h = logsumexp(log_posterior1, axis=1)

print(f"Log Marginal MAP: x_lh = {x_lhs[np.argmax(log_marg_x)]}")
print(f"Log Marginal MAP: h = {hs[np.argmax(log_marg_h)]}")

## Problem 3.1

Write down a function to compute an array containing the log of the sampling distribution using an array of flash positions, not just single point.

In [ ]:
def log_p_many(x_lh, h_lh, flash):
    log_prob = np.zeros_like(x_lh)
    for f in flash:
        log_prob += np.log(h_lh) - np.log(np.pi) - np.log((f - x_lh)**2 + h_lh**2)
    return log_prob

## Problem 3.2

Plot the (log)posterior distribution after two flashes. Calculate the MAP estimate, as well as the marginal distributions and MAP estimates for each variable separately.

In [ ]:
log_p_array2 = log_p_many(x_lh_grid, h_grid, flash_x[:2])
log_posterior2 = log_p_array2 + log_prior_array
log_posterior2 -= logsumexp(log_posterior2)

map_idx2 = np.unravel_index(np.argmax(log_posterior2), log_posterior2.shape)
print(f"Joint MAP (2 flashes): x_lh = {x_lhs[map_idx2[1]]}, h = {hs[map_idx2[0]]}")

log_marg_x2 = logsumexp(log_posterior2, axis=0)
log_marg_h2 = logsumexp(log_posterior2, axis=1)

print(f"Marginal MAP x_lh: {x_lhs[np.argmax(log_marg_x2)]}")
print(f"Marginal MAP h: {hs[np.argmax(log_marg_h2)]}")

levels = 20
max_log_posterior2 = log_posterior2.max()
cs = plt.contourf(x_lhs, hs, log_posterior2 - max_log_posterior2, levels=levels)
plt.contour(x_lhs, hs, log_posterior2 - max_log_posterior2, colors='black', linewidths=0.5, levels=levels)
plt.colorbar(cs)
plt.xlabel('x_lh')
plt.ylabel('h')
plt.show()

plt.plot(x_lhs, np.exp(log_marg_x2))
plt.xlabel('x_lh')
plt.ylabel('Marginal Probability')
plt.show()

plt.plot(hs, np.exp(log_marg_h2))
plt.xlabel('h')
plt.ylabel('Marginal Probability')
plt.show()

## Problem 3.3

Perform same calculations as in the previous problem, but use all the flash positions.

In [ ]:
log_p_array_all = log_p_many(x_lh_grid, h_grid, flash_x)
log_posterior_all = log_p_array_all + log_prior_array
log_posterior_all -= logsumexp(log_posterior_all)

map_idx_all = np.unravel_index(np.argmax(log_posterior_all), log_posterior_all.shape)
print(f"Joint MAP (all flashes): x_lh = {x_lhs[map_idx_all[1]]}, h = {hs[map_idx_all[0]]}")

log_marg_x_all = logsumexp(log_posterior_all, axis=0)
log_marg_h_all = logsumexp(log_posterior_all, axis=1)

print(f"Marginal MAP x_lh: {x_lhs[np.argmax(log_marg_x_all)]}")
print(f"Marginal MAP h: {hs[np.argmax(log_marg_h_all)]}")

levels = 20
max_log_posterior_all = log_posterior_all.max()
cs = plt.contourf(x_lhs, hs, log_posterior_all - max_log_posterior_all, levels=levels)
plt.contour(x_lhs, hs, log_posterior_all - max_log_posterior_all, colors='black', linewidths=0.5, levels=levels)
plt.colorbar(cs)
plt.xlabel('x_lh')
plt.ylabel('h')
plt.show()

plt.plot(x_lhs, np.exp(log_marg_x_all))
plt.xlabel('x_lh')
plt.ylabel('Marginal Probability')
plt.show()

plt.plot(hs, np.exp(log_marg_h_all))
plt.xlabel('h')
plt.ylabel('Marginal Probability')
plt.show()